# proTES and GA4GH TES API demo

This notebook demonstrates the **proTES (Proxy Task Execution Service)** and the **GA4GH Task Execution Service (TES) API**.

## What is proTES?
proTES is a proxy gateway for the GA4GH TES API. It provides:
- Task distribution across multiple TES endpoints
- Middleware support for custom task processing
- Load balancing strategies (random and distance-based)
- Task tracking and status monitoring
- Centralized access to distributed compute resources

## Overview
1. Interacting with TES API endpoints
2. Creating, submitting, and monitoring tasks
3. Service information and capabilities
4. Task management and lifecycle operations
5. Examples with containerized workloads

In [ ]:
# Import Required Libraries
import requests
import json
import time
import pandas as pd
from datetime import datetime
from typing import Dict, List, Optional
import uuid

# Configuration
PROTES_BASE_URL = "http://localhost:8080"
TES_API_BASE = f"{PROTES_BASE_URL}/ga4gh/tes/v1"

# Helper function for making API requests
def make_request(method: str, endpoint: str, data: Optional[Dict] = None, params: Optional[Dict] = None) -> Dict:
    """
    Make HTTP request to proTES API
    """
    url = f"{TES_API_BASE}/{endpoint}"
    
    try:
        if method.upper() == 'GET':
            response = requests.get(url, params=params)
        elif method.upper() == 'POST':
            response = requests.post(url, json=data, params=params)
        elif method.upper() == 'DELETE':
            response = requests.delete(url)
        
        response.raise_for_status()
        return response.json() if response.content else {}
    except requests.exceptions.RequestException as e:
        print(f"API request failed: {e}")
        if hasattr(e, 'response') and e.response is not None:
            try:
                error_detail = e.response.json()
                print(f"Error details: {json.dumps(error_detail, indent=2)}")
            except:
                print(f"Response content: {e.response.text}")
        return {}

print("Libraries imported and API helper configured.")
print(f"proTES API Base URL: {TES_API_BASE}")

## 1. Service Discovery & Information

First, let's check if proTES is running and get service information, including supported features and configurations.

In [ ]:
# Check if proTES service is accessible
def check_service_health():
    """Check if proTES service is running and accessible"""
    try:
        response = requests.get(f"{PROTES_BASE_URL}/ga4gh/tes/v1/ui/")
        if response.status_code == 200:
            print("proTES service is running and accessible.")
            print(f"Swagger UI available at: {PROTES_BASE_URL}/ga4gh/tes/v1/ui/")
            return True
        else:
            print(f"Service check failed with status: {response.status_code}")
            return False
    except Exception as e:
        print(f"Cannot connect to proTES service: {e}")
        return False

# Get service information
def get_service_info():
    """Retrieve TES service information"""
    print("Attempting to get service information...")
    
    # Try different possible endpoints for service info
    endpoints = ["service-info", "serviceinfo", "service_info"]
    
    for endpoint in endpoints:
        try:
            response = requests.get(f"{TES_API_BASE}/{endpoint}")
            if response.status_code == 200:
                service_info = response.json()
                print(f"Service info retrieved from endpoint: {endpoint}")
                return service_info
        except Exception as e:
            continue
    
    print("Service info endpoint may not be implemented or may require authentication.")
    return None

# Check service health
service_healthy = check_service_health()

if service_healthy:
    service_info = get_service_info()
    if service_info:
        print("\n📋 Service Information:")
        print(json.dumps(service_info, indent=2))
    else:
        print("ℹ️  Continuing with other API endpoints...")

## 2. Task Creation and Submission

The core functionality of TES is task execution. Let's create and submit different types of tasks to demonstrate the API capabilities.

In [ ]:
def create_simple_task():
    """
    Create a simple 'Hello World' task using a basic container
    """
    task = {
        "name": "Hello World Task",
        "description": "A simple hello world task to test TES functionality",
        "executors": [
            {
                "image": "alpine:latest",
                "command": ["echo", "Hello from proTES!"],
                "workdir": "/tmp",
                "stdout": "/tmp/stdout.log",
                "stderr": "/tmp/stderr.log"
            }
        ],
        "outputs": [
            {
                "name": "stdout_file",
                "path": "/tmp/stdout.log",
                "type": "FILE"
            }
        ],
        "tags": {
            "task_type": "demo",
            "complexity": "simple"
        }
    }
    return task

def create_compute_task():
    """
    Create a computational task that performs some basic operations
    """
    task = {
        "name": "Basic Computation Task",
        "description": "A task that performs basic mathematical operations",
        "executors": [
            {
                "image": "python:3.9-slim",
                "command": [
                    "python3", "-c",
                    """
import math
import time
print('Starting computation...')
result = sum(i**2 for i in range(1000))
print(f'Sum of squares 1-1000: {result}')
print(f'Square root of result: {math.sqrt(result)}')
time.sleep(2)
print('Computation complete!')
                    """
                ],
                "workdir": "/tmp",
                "stdout": "/tmp/computation.log",
                "stderr": "/tmp/computation.err"
            }
        ],
        "outputs": [
            {
                "name": "computation_output",
                "path": "/tmp/computation.log",
                "type": "FILE"
            }
        ],
        "resources": {
            "cpu_cores": 1,
            "ram_gb": 1.0,
            "disk_gb": 1.0
        },
        "tags": {
            "task_type": "computation",
            "complexity": "medium"
        }
    }
    return task

def create_data_processing_task():
    """
    Create a data processing task that works with files
    """
    task = {
        "name": "Data Processing Task",
        "description": "A task that processes and analyzes data",
        "inputs": [
            {
                "name": "input_data",
                "description": "Input data file",
                "path": "/data/input.txt",
                "type": "FILE",
                "content": "name,age,city\nAlice,30,New York\nBob,25,San Francisco\nCharlie,35,Chicago\nDiana,28,Boston"
            }
        ],
        "executors": [
            {
                "image": "python:3.9-slim",
                "command": [
                    "python3", "-c",
                    """
import csv
import json

# Read input data
with open('/data/input.txt', 'r') as f:
    reader = csv.DictReader(f)
    data = list(reader)

print(f'Processing {len(data)} records...')

# Process data
total_age = sum(int(person['age']) for person in data)
average_age = total_age / len(data)
cities = list(set(person['city'] for person in data))

# Create results
results = {
    'total_records': len(data),
    'average_age': round(average_age, 2),
    'unique_cities': cities,
    'oldest_person': max(data, key=lambda x: int(x['age']))
}

print('Results:', json.dumps(results, indent=2))

# Save results
with open('/output/results.json', 'w') as f:
    json.dump(results, f, indent=2)

print('Data processing complete!')
                    """
                ],
                "workdir": "/tmp",
                "stdout": "/output/processing.log",
                "stderr": "/output/processing.err"
            }
        ],
        "outputs": [
            {
                "name": "results_file",
                "path": "/output/results.json",
                "type": "FILE"
            },
            {
                "name": "processing_log",
                "path": "/output/processing.log", 
                "type": "FILE"
            }
        ],
        "resources": {
            "cpu_cores": 1,
            "ram_gb": 2.0,
            "disk_gb": 5.0
        },
        "tags": {
            "task_type": "data_processing",
            "complexity": "advanced",
            "language": "python"
        }
    }
    return task

print("✓ Task creation functions defined!")
print("Available task types:")
print("  • Simple Hello World task")
print("  • Computational task with mathematical operations")
print("  • Data processing task with file I/O")

In [ ]:
def submit_task(task_definition: Dict) -> Optional[str]:
    """
    Submit a task to proTES and return the task ID
    """
    print(f"→ Submitting task: {task_definition.get('name', 'Unnamed Task')}")
    print(f"📄 Description: {task_definition.get('description', 'No description')}")
    
    try:
        response = requests.post(f"{TES_API_BASE}/tasks", json=task_definition)
        
        if response.status_code == 200:
            result = response.json()
            task_id = result.get('id')
            print(f"✓ Task submitted successfully!")
            print(f"🆔 Task ID: {task_id}")
            return task_id
        else:
            print(f"❌ Task submission failed with status: {response.status_code}")
            try:
                error_detail = response.json()
                print(f"Error details: {json.dumps(error_detail, indent=2)}")
            except:
                print(f"Response content: {response.text}")
            return None
            
    except Exception as e:
        print(f"❌ Exception during task submission: {e}")
        return None

# Let's create and submit a simple task first
print("🔬 Creating and submitting a simple Hello World task...")
simple_task = create_simple_task()

# Display the task definition
print("\n📋 Task Definition:")
print(json.dumps(simple_task, indent=2))

# Submit the task
task_id = submit_task(simple_task)

## 3. Task Monitoring and Management

After submitting tasks, we need to monitor their progress and manage their lifecycle. TES provides endpoints for querying task status, retrieving task details, and managing task execution.

In [ ]:
def get_task_details(task_id: str, view: str = "BASIC") -> Optional[Dict]:
    """
    Get detailed information about a specific task
    
    Args:
        task_id: The ID of the task to query
        view: Level of detail (MINIMAL, BASIC, FULL)
    """
    if not task_id:
        print("❌ No task ID provided")
        return None
        
    try:
        params = {"view": view}
        response = requests.get(f"{TES_API_BASE}/tasks/{task_id}", params=params)
        
        if response.status_code == 200:
            task_details = response.json()
            print(f"✓ Retrieved task details for: {task_id}")
            return task_details
        else:
            print(f"❌ Failed to get task details. Status: {response.status_code}")
            try:
                error = response.json()
                print(f"Error: {json.dumps(error, indent=2)}")
            except:
                print(f"Response: {response.text}")
            return None
    except Exception as e:
        print(f"❌ Exception getting task details: {e}")
        return None

def list_tasks(name_prefix: str = None, page_size: int = 10, page_token: str = None) -> Optional[Dict]:
    """
    List tasks with optional filtering
    
    Args:
        name_prefix: Filter tasks by name prefix
        page_size: Number of tasks per page
        page_token: Token for pagination
    """
    try:
        params = {"page_size": page_size}
        if name_prefix:
            params["name_prefix"] = name_prefix
        if page_token:
            params["page_token"] = page_token
            
        response = requests.get(f"{TES_API_BASE}/tasks", params=params)
        
        if response.status_code == 200:
            tasks_list = response.json()
            print(f"✓ Retrieved task list")
            return tasks_list
        else:
            print(f"❌ Failed to list tasks. Status: {response.status_code}")
            try:
                error = response.json()
                print(f"Error: {json.dumps(error, indent=2)}")
            except:
                print(f"Response: {response.text}")
            return None
    except Exception as e:
        print(f"❌ Exception listing tasks: {e}")
        return None

def cancel_task(task_id: str) -> bool:
    """
    Cancel a running task
    """
    if not task_id:
        print("❌ No task ID provided")
        return False
        
    try:
        response = requests.post(f"{TES_API_BASE}/tasks/{task_id}:cancel")
        
        if response.status_code == 200:
            print(f"✓ Task {task_id} cancelled successfully")
            return True
        else:
            print(f"❌ Failed to cancel task. Status: {response.status_code}")
            return False
    except Exception as e:
        print(f"❌ Exception cancelling task: {e}")
        return False

def monitor_task_progress(task_id: str, max_checks: int = 30, check_interval: int = 5):
    """
    Monitor a task's progress until completion or timeout
    """
    if not task_id:
        print("❌ No task ID provided")
        return
        
    print(f"🔎 Monitoring task progress: {task_id}")
    print(f"⏱️  Will check every {check_interval} seconds (max {max_checks} checks)")
    
    for check in range(max_checks):
        task_details = get_task_details(task_id, view="BASIC")
        
        if not task_details:
            print("❌ Failed to get task details")
            break
            
        state = task_details.get('state', 'UNKNOWN')
        task_name = task_details.get('name', 'Unknown Task')
        
        print(f"📈 Check {check + 1}/{max_checks} - Task: {task_name}, State: {state}")
        
        # Terminal states
        if state in ['COMPLETE', 'EXECUTOR_ERROR', 'SYSTEM_ERROR', 'CANCELED']:
            print(f"🏁 Task reached terminal state: {state}")
            
            # Show task logs if available
            if 'logs' in task_details:
                print("\n📄 Task Logs:")
                for i, log in enumerate(task_details['logs']):
                    print(f"  Executor {i + 1}:")
                    if 'stdout' in log:
                        print(f"    stdout: {log['stdout']}")
                    if 'stderr' in log:
                        print(f"    stderr: {log['stderr']}")
                    if 'exit_code' in log:
                        print(f"    exit_code: {log['exit_code']}")
            
            return task_details
            
        # Continue monitoring
        if check < max_checks - 1:
            time.sleep(check_interval)
    
    print(f"⏰ Monitoring timeout reached after {max_checks} checks")
    return task_details

print("✓ Task monitoring functions defined!")
print("Available monitoring operations:")
print("  • Get detailed task information")
print("  • List all tasks with filtering")
print("  • Cancel running tasks")
print("  • Monitor task progress in real-time")

In [ ]:
# Let's demonstrate task monitoring by checking on our submitted task
if 'task_id' in locals() and task_id:
    print("🔎 Checking status of our submitted task...")
    
    # Get basic task details
    task_details = get_task_details(task_id, view="BASIC")
    
    if task_details:
        print(f"\n📋 Task Status Summary:")
        print(f"  Name: {task_details.get('name', 'N/A')}")
        print(f"  State: {task_details.get('state', 'N/A')}")
        print(f"  Creation Time: {task_details.get('creation_time', 'N/A')}")
        
        # If task is not in a terminal state, monitor it
        current_state = task_details.get('state', 'UNKNOWN')
        if current_state not in ['COMPLETE', 'EXECUTOR_ERROR', 'SYSTEM_ERROR', 'CANCELED']:
            print(f"\n🔄 Task is in {current_state} state. Starting monitoring...")
            final_details = monitor_task_progress(task_id, max_checks=10, check_interval=3)
        else:
            print(f"\n✓ Task is already in terminal state: {current_state}")
else:
    print("ℹ️  No task ID available from previous submission. Let's list existing tasks...")
    
    # List existing tasks
    tasks_response = list_tasks(page_size=5)
    if tasks_response and 'tasks' in tasks_response:
        tasks = tasks_response['tasks']
        print(f"\n📄 Found {len(tasks)} recent tasks:")
        
        for i, task in enumerate(tasks, 1):
            print(f"  {i}. {task.get('name', 'Unnamed')} (ID: {task.get('id', 'N/A')}) - State: {task.get('state', 'N/A')}")
    else:
        print("📭 No tasks found or unable to retrieve task list")

## 4. Advanced Task Examples

Now let's explore more complex task scenarios that demonstrate the full capabilities of the TES API, including data processing, multi-step workflows, and resource management.

In [ ]:
# Let's submit and monitor a more complex computational task
print("→ Creating and submitting a computational task...")
compute_task = create_compute_task()

print("\n📋 Computational Task Definition:")
print(json.dumps(compute_task, indent=2))

# Submit the computational task
compute_task_id = submit_task(compute_task)

if compute_task_id:
    print(f"\n🔎 Monitoring computational task: {compute_task_id}")
    final_compute_details = monitor_task_progress(compute_task_id, max_checks=15, check_interval=3)
    
    if final_compute_details:
        print(f"\n📈 Final task details:")
        print(f"  State: {final_compute_details.get('state', 'N/A')}")
        print(f"  Resources used: {final_compute_details.get('resources', 'N/A')}")

print("\n" + "="*50)

# Now let's try a data processing task
print("\n→ Creating and submitting a data processing task...")
data_task = create_data_processing_task()

print("\n📋 Data Processing Task Definition:")
print(json.dumps(data_task, indent=2))

# Submit the data processing task
data_task_id = submit_task(data_task)

if data_task_id:
    print(f"\n🔎 Monitoring data processing task: {data_task_id}")
    final_data_details = monitor_task_progress(data_task_id, max_checks=15, check_interval=3)
    
    if final_data_details:
        print(f"\n📈 Final task details:")
        print(f"  State: {final_data_details.get('state', 'N/A')}")
        print(f"  Outputs: {len(final_data_details.get('outputs', []))} files")

## 5. Task Analytics and Reporting

Let's analyze the tasks we've submitted and create some basic reports on their performance and status.

In [ ]:
def generate_task_report(task_ids: List[str]) -> pd.DataFrame:
    """
    Generate a comprehensive report for a list of tasks
    """
    task_data = []
    
    for task_id in task_ids:
        if not task_id:
            continue
            
        task_details = get_task_details(task_id, view="FULL")
        if not task_details:
            continue
            
        # Extract key information
        task_info = {
            'task_id': task_id,
            'name': task_details.get('name', 'Unknown'),
            'state': task_details.get('state', 'Unknown'),
            'creation_time': task_details.get('creation_time', ''),
            'start_time': task_details.get('start_time', ''),
            'end_time': task_details.get('end_time', ''),
            'cpu_cores': task_details.get('resources', {}).get('cpu_cores', 0),
            'ram_gb': task_details.get('resources', {}).get('ram_gb', 0),
            'disk_gb': task_details.get('resources', {}).get('disk_gb', 0),
            'num_executors': len(task_details.get('executors', [])),
            'num_inputs': len(task_details.get('inputs', [])),
            'num_outputs': len(task_details.get('outputs', [])),
            'task_type': task_details.get('tags', {}).get('task_type', 'unknown'),
            'complexity': task_details.get('tags', {}).get('complexity', 'unknown')
        }
        
        # Calculate duration if possible
        if task_info['start_time'] and task_info['end_time']:
            try:
                start = datetime.fromisoformat(task_info['start_time'].replace('Z', '+00:00'))
                end = datetime.fromisoformat(task_info['end_time'].replace('Z', '+00:00'))
                duration = (end - start).total_seconds()
                task_info['duration_seconds'] = duration
            except:
                task_info['duration_seconds'] = None
        else:
            task_info['duration_seconds'] = None
            
        task_data.append(task_info)
    
    return pd.DataFrame(task_data)

def analyze_task_performance(df: pd.DataFrame):
    """
    Analyze task performance metrics
    """
    if df.empty:
        print("📈 No task data available for analysis")
        return
        
    print("📈 Task Performance Analysis")
    print("=" * 40)
    
    # Basic statistics
    total_tasks = len(df)
    print(f"Total Tasks: {total_tasks}")
    
    # State distribution
    state_counts = df['state'].value_counts()
    print(f"\n📈 Task States:")
    for state, count in state_counts.items():
        percentage = (count / total_tasks) * 100
        print(f"  {state}: {count} ({percentage:.1f}%)")
    
    # Task type distribution
    if 'task_type' in df.columns:
        type_counts = df['task_type'].value_counts()
        print(f"\n🏷️  Task Types:")
        for task_type, count in type_counts.items():
            percentage = (count / total_tasks) * 100
            print(f"  {task_type}: {count} ({percentage:.1f}%)")
    
    # Resource analysis
    if df['cpu_cores'].sum() > 0:
        print(f"\n💻 Resource Usage:")
        print(f"  Total CPU cores requested: {df['cpu_cores'].sum()}")
        print(f"  Average CPU cores per task: {df['cpu_cores'].mean():.2f}")
        print(f"  Total RAM requested: {df['ram_gb'].sum():.2f} GB")
        print(f"  Average RAM per task: {df['ram_gb'].mean():.2f} GB")
    
    # Duration analysis
    completed_tasks = df[df['duration_seconds'].notna()]
    if not completed_tasks.empty:
        print(f"\n⏱️  Execution Time Analysis:")
        print(f"  Completed tasks: {len(completed_tasks)}")
        print(f"  Average duration: {completed_tasks['duration_seconds'].mean():.2f} seconds")
        print(f"  Fastest task: {completed_tasks['duration_seconds'].min():.2f} seconds")
        print(f"  Slowest task: {completed_tasks['duration_seconds'].max():.2f} seconds")

# Collect all task IDs from our session
submitted_task_ids = []
if 'task_id' in locals() and task_id:
    submitted_task_ids.append(task_id)
if 'compute_task_id' in locals() and compute_task_id:
    submitted_task_ids.append(compute_task_id)
if 'data_task_id' in locals() and data_task_id:
    submitted_task_ids.append(data_task_id)

if submitted_task_ids:
    print(f"📋 Generating report for {len(submitted_task_ids)} submitted tasks...")
    
    # Generate task report
    task_df = generate_task_report(submitted_task_ids)
    
    if not task_df.empty:
        print("\n📈 Task Summary Table:")
        print(task_df[['name', 'state', 'task_type', 'cpu_cores', 'ram_gb']].to_string(index=False))
        
        # Analyze performance
        print("\n")
        analyze_task_performance(task_df)
    else:
        print("❌ No task data retrieved for analysis")
else:
    print("ℹ️  No tasks were submitted in this session to analyze")
    
    # Try to get some tasks from the system
    print("\n🔎 Attempting to retrieve recent tasks from the system...")
    recent_tasks = list_tasks(page_size=10)
    
    if recent_tasks and 'tasks' in recent_tasks:
        task_list = recent_tasks['tasks']
        if task_list:
            recent_task_ids = [task.get('id') for task in task_list if task.get('id')]
            print(f"📋 Found {len(recent_task_ids)} recent tasks. Generating report...")
            
            task_df = generate_task_report(recent_task_ids[:5])  # Limit to 5 for demo
            if not task_df.empty:
                print("\n📈 Recent Tasks Summary:")
                print(task_df[['name', 'state', 'task_type']].to_string(index=False))
                analyze_task_performance(task_df)

## 6. proTES Middleware and Configuration

proTES provides powerful middleware capabilities for task distribution and processing. Let's explore how tasks are routed and distributed across different TES endpoints.

In [ ]:
def explore_protes_configuration():
    """
    Explore proTES configuration and middleware setup
    """
    print("🔧 proTES Configuration Overview")
    print("=" * 40)
    
    # Read the configuration file
    try:
        with open('config.yaml', 'r') as f:
            import yaml
            config = yaml.safe_load(f)
            
        print("✓ Successfully loaded proTES configuration")
        
        # Service endpoints
        if 'tes' in config and 'service_list' in config['tes']:
            endpoints = config['tes']['service_list']
            print(f"\n🌐 Configured TES Endpoints ({len(endpoints)}):")
            for i, endpoint in enumerate(endpoints, 1):
                print(f"  {i}. {endpoint}")
        
        # Middleware configuration
        if 'middlewares' in config:
            middlewares = config['middlewares']
            print(f"\n🔧 Configured Middlewares:")
            for i, middleware_group in enumerate(middlewares, 1):
                print(f"  Group {i}:")
                for middleware in middleware_group:
                    middleware_name = middleware.split('.')[-1]
                    print(f"    • {middleware_name}")
        
        # Service info
        if 'serviceInfo' in config:
            service_info = config['serviceInfo']
            print(f"\n📋 Service Information:")
            print(f"  Name: {service_info.get('name', 'N/A')}")
            print(f"  Description: {service_info.get('doc', 'N/A')}")
            
        # Database configuration
        if 'db' in config:
            db_config = config['db']
            print(f"\n🗄️  Database Configuration:")
            print(f"  Host: {db_config.get('host', 'N/A')}")
            print(f"  Port: {db_config.get('port', 'N/A')}")
            
        # Jobs/Queue configuration
        if 'jobs' in config:
            jobs_config = config['jobs']
            print(f"\n📋 Job Queue Configuration:")
            print(f"  Host: {jobs_config.get('host', 'N/A')}")
            print(f"  Port: {jobs_config.get('port', 'N/A')}")
            
    except FileNotFoundError:
        print("❌ Configuration file 'config.yaml' not found in current directory")
        print("ℹ️  This is expected when running outside the proTES directory")
    except Exception as e:
        print(f"❌ Error reading configuration: {e}")

def demonstrate_task_distribution():
    """
    Demonstrate how proTES distributes tasks across endpoints
    """
    print("\n→ Task Distribution Demonstration")
    print("=" * 40)
    
    print("proTES uses middleware to determine where tasks should be executed:")
    print("\n🎯 Built-in Distribution Strategies:")
    print("  1. Random Distribution:")
    print("     - Randomly selects from available TES endpoints")
    print("     - Provides simple load balancing")
    print("     - Good for homogeneous compute environments")
    
    print("\n  2. Distance-based Distribution:")
    print("     - Considers geographic distance to data")
    print("     - Minimizes data transfer overhead")
    print("     - Optimal for data-intensive workloads")
    
    print("\n📄 Middleware Chain:")
    print("  • Tasks are submitted to proTES")
    print("  • Middleware evaluates task requirements")
    print("  • Best endpoint is selected based on strategy")
    print("  • Task is forwarded to chosen TES endpoint")
    print("  • proTES tracks and monitors execution")
    
    # Create multiple tasks to demonstrate distribution
    distribution_tasks = []
    
    for i in range(3):
        task = {
            "name": f"Distribution Test Task {i+1}",
            "description": f"Task {i+1} to demonstrate proTES distribution",
            "executors": [
                {
                    "image": "alpine:latest",
                    "command": ["echo", f"Hello from distributed task {i+1}!"],
                    "stdout": "/tmp/output.log"
                }
            ],
            "tags": {
                "test_type": "distribution",
                "task_number": str(i+1)
            }
        }
        distribution_tasks.append(task)
    
    print(f"\n🔬 Creating {len(distribution_tasks)} tasks to demonstrate distribution...")
    
    task_ids = []
    for i, task in enumerate(distribution_tasks):
        print(f"\n📤 Submitting task {i+1}...")
        task_id = submit_task(task)
        if task_id:
            task_ids.append(task_id)
            # Small delay between submissions
            time.sleep(1)
    
    if task_ids:
        print(f"\n✓ Successfully submitted {len(task_ids)} tasks for distribution")
        print("🔎 These tasks may be distributed across different TES endpoints")
        print("📈 proTES middleware will handle the routing and load balancing")
        
        return task_ids
    else:
        print("❌ No tasks were successfully submitted")
        return []

# Explore configuration
explore_protes_configuration()

# Demonstrate task distribution
distribution_task_ids = demonstrate_task_distribution()

## 7. Error Handling and Troubleshooting

Understanding how to handle errors and troubleshoot issues is crucial when working with distributed task execution systems.

In [ ]:
def create_error_task():
    """
    Create a task that will intentionally fail to demonstrate error handling
    """
    task = {
        "name": "Intentional Error Task",
        "description": "A task designed to fail for error handling demonstration",
        "executors": [
            {
                "image": "alpine:latest",
                "command": ["exit", "1"],  # This will cause the task to fail
                "workdir": "/tmp",
                "stdout": "/tmp/stdout.log",
                "stderr": "/tmp/stderr.log"
            }
        ],
        "tags": {
            "task_type": "error_demo",
            "expected_outcome": "failure"
        }
    }
    return task

def create_resource_intensive_task():
    """
    Create a task with very high resource requirements to test limits
    """
    task = {
        "name": "Resource Intensive Task",
        "description": "A task with high resource requirements",
        "executors": [
            {
                "image": "python:3.9-slim",
                "command": ["python3", "-c", "print('Resource intensive task')"],
                "workdir": "/tmp"
            }
        ],
        "resources": {
            "cpu_cores": 100,  # Unrealistic requirement
            "ram_gb": 1000.0,  # Very high memory requirement
            "disk_gb": 10000.0  # Very high disk requirement
        },
        "tags": {
            "task_type": "resource_test",
            "complexity": "extreme"
        }
    }
    return task

def demonstrate_error_handling():
    """
    Demonstrate various error scenarios and how to handle them
    """
    print("⚠️  Error Handling and Troubleshooting Demo")
    print("=" * 50)
    
    # Test 1: Task with execution error
    print("\n🔬 Test 1: Task with Execution Error")
    print("-" * 30)
    
    error_task = create_error_task()
    error_task_id = submit_task(error_task)
    
    if error_task_id:
        print("🔎 Monitoring error task...")
        error_details = monitor_task_progress(error_task_id, max_checks=10, check_interval=2)
        
        if error_details:
            state = error_details.get('state')
            print(f"\n📈 Error Task Result: {state}")
            
            if state == 'EXECUTOR_ERROR':
                print("✓ Successfully demonstrated executor error handling")
                
                # Show logs if available
                logs = error_details.get('logs', [])
                if logs:
                    for i, log in enumerate(logs):
                        print(f"\n📄 Executor {i+1} logs:")
                        if 'exit_code' in log:
                            print(f"  Exit code: {log['exit_code']}")
                        if 'stderr' in log:
                            print(f"  Stderr: {log['stderr']}")
    
    # Test 2: Invalid task submission
    print("\n🔬 Test 2: Invalid Task Submission")
    print("-" * 30)
    
    invalid_task = {
        "name": "Invalid Task",
        "description": "Task with invalid structure",
        "executors": [
            {
                # Missing required 'image' field
                "command": ["echo", "This will fail"]
            }
        ]
    }
    
    print("📤 Attempting to submit invalid task...")
    invalid_task_id = submit_task(invalid_task)
    
    if not invalid_task_id:
        print("✓ Successfully demonstrated invalid task rejection")
    
    # Test 3: Resource constraint handling
    print("\n🔬 Test 3: Resource Constraint Test")
    print("-" * 30)
    
    resource_task = create_resource_intensive_task()
    print("📤 Submitting resource-intensive task...")
    resource_task_id = submit_task(resource_task)
    
    if resource_task_id:
        print("🔎 Monitoring resource-intensive task...")
        resource_details = monitor_task_progress(resource_task_id, max_checks=8, check_interval=2)
        
        if resource_details:
            state = resource_details.get('state')
            print(f"📈 Resource Task Result: {state}")
            
            if state in ['SYSTEM_ERROR', 'EXECUTOR_ERROR']:
                print("✓ Task failed due to resource constraints (as expected)")
    
    # Test 4: Connectivity test
    print("\n🔬 Test 4: Service Connectivity")
    print("-" * 30)
    
    try:
        # Test with a clearly invalid endpoint
        invalid_url = "http://localhost:9999/invalid/endpoint"
        response = requests.get(invalid_url, timeout=2)
    except requests.exceptions.RequestException as e:
        print(f"✓ Successfully demonstrated connection error handling: {type(e).__name__}")
    
    print("\n💡 Troubleshooting Tips:")
    print("=" * 30)
    print("1. Check task logs for execution errors")
    print("2. Verify resource requirements are reasonable")
    print("3. Ensure container images are accessible")
    print("4. Monitor task state transitions")
    print("5. Use appropriate timeout values")
    print("6. Check proTES service logs for system issues")

# Common error patterns and solutions
def show_common_errors():
    """
    Display common error patterns and their solutions
    """
    print("\n🔧 Common TES Error Patterns and Solutions")
    print("=" * 50)
    
    errors = [
        {
            "error": "404 Not Found",
            "cause": "Endpoint doesn't exist or wrong URL",
            "solution": "Check the API base URL and endpoint path"
        },
        {
            "error": "EXECUTOR_ERROR with exit code 1",
            "cause": "Command execution failed",
            "solution": "Check command syntax and container image"
        },
        {
            "error": "SYSTEM_ERROR",
            "cause": "Infrastructure or resource issues",
            "solution": "Check resource requirements and system capacity"
        },
        {
            "error": "Task stuck in QUEUED state",
            "cause": "No available compute resources",
            "solution": "Wait for resources or adjust requirements"
        },
        {
            "error": "Connection timeout",
            "cause": "Network issues or overloaded service",
            "solution": "Retry with backoff or check service status"
        }
    ]
    
    for i, error_info in enumerate(errors, 1):
        print(f"\n{i}. {error_info['error']}")
        print(f"   Cause: {error_info['cause']}")
        print(f"   Solution: {error_info['solution']}")

# Run error handling demonstration
demonstrate_error_handling()
show_common_errors()

## 8. Best Practices and Performance Optimization

This section covers best practices for designing efficient TES tasks and optimizing performance when working with proTES.

In [ ]:
def demonstrate_optimal_task_design():
    """
    Demonstrate best practices for task design and optimization
    """
    print("→ TES Task Design Best Practices")
    print("=" * 40)
    
    # Example 1: Optimized single-executor task
    optimized_task = {
        "name": "Optimized Data Processing",
        "description": "Well-designed task with proper resource allocation and output handling",
        "executors": [
            {
                "image": "python:3.9-slim",
                "command": [
                    "python3", "-c", """
import time
import os

# Efficient data processing simulation
print('Starting optimized data processing...')
start_time = time.time()

# Simulate processing with progress updates
for i in range(1, 11):
    print(f'Processing batch {i}/10 ({i*10}% complete)')
    time.sleep(0.5)

end_time = time.time()
print(f'Processing completed in {end_time - start_time:.2f} seconds')

# Write results to output file
with open('/tmp/results.txt', 'w') as f:
    f.write('Processing completed successfully\\n')
    f.write(f'Total time: {end_time - start_time:.2f} seconds\\n')
    f.write('Status: SUCCESS\\n')
"""
                ],
                "workdir": "/tmp",
                "stdout": "/tmp/stdout.log",
                "stderr": "/tmp/stderr.log"
            }
        ],
        "inputs": [],
        "outputs": [
            {
                "name": "results",
                "url": "file:///tmp/results.txt",
                "path": "/tmp/results.txt"
            }
        ],
        "resources": {
            "cpu_cores": 1,
            "ram_gb": 1.0,
            "disk_gb": 1.0
        },
        "tags": {
            "task_type": "data_processing",
            "optimization": "enabled",
            "priority": "normal"
        }
    }
    
    print("📄 Optimized Task Structure:")
    print(f"  ✓ Clear name and description")
    print(f"  ✓ Appropriate resource allocation")
    print(f"  ✓ Progress reporting in logs")
    print(f"  ✓ Proper output file handling")
    print(f"  ✓ Meaningful tags for categorization")
    
    # Submit and monitor the optimized task
    print("\n📤 Submitting optimized task...")
    task_id = submit_task(optimized_task)
    
    if task_id:
        print(f"✓ Task submitted with ID: {task_id}")
        
        # Monitor with appropriate intervals
        result = monitor_task_progress(task_id, max_checks=20, check_interval=2)
        
        if result and result.get('state') == 'COMPLETE':
            print("🎉 Optimized task completed successfully!")
            
            # Demonstrate output retrieval
            outputs = result.get('outputs', [])
            if outputs:
                print(f"📁 Generated {len(outputs)} output file(s)")
                for output in outputs:
                    print(f"  📄 {output.get('name', 'unnamed')}: {output.get('url', 'no URL')}")
    
    return optimized_task

def show_performance_tips():
    """
    Display performance optimization tips
    """
    print("\n⚡ Performance Optimization Tips")
    print("=" * 40)
    
    tips = [
        {
            "category": "Resource Allocation",
            "tips": [
                "Start with minimal resources and scale up as needed",
                "Use appropriate CPU cores for your workload type",
                "Allocate sufficient RAM to avoid out-of-memory errors",
                "Consider disk I/O requirements for data-intensive tasks"
            ]
        },
        {
            "category": "Container Images",
            "tips": [
                "Use smaller, specialized base images (alpine, slim variants)",
                "Pre-install dependencies in custom images",
                "Use image caching to speed up task startup",
                "Avoid pulling large images for simple tasks"
            ]
        },
        {
            "category": "Task Design",
            "tips": [
                "Break large tasks into smaller, parallel subtasks",
                "Use appropriate polling intervals for monitoring",
                "Implement proper error handling and recovery",
                "Include progress reporting in task output"
            ]
        },
        {
            "category": "Data Management",
            "tips": [
                "Minimize data transfer by processing data locally",
                "Use efficient file formats (parquet, HDF5)",
                "Implement proper input/output file handling",
                "Consider data locality for distributed tasks"
            ]
        }
    ]
    
    for tip_category in tips:
        print(f"\n🎯 {tip_category['category']}:")
        for tip in tip_category['tips']:
            print(f"  • {tip}")

def demonstrate_batch_optimization():
    """
    Show how to optimize batch task processing
    """
    print("\n📦 Batch Processing Optimization")
    print("=" * 40)
    
    # Example: Processing multiple files efficiently
    batch_tasks = []
    
    for i in range(1, 4):  # Create 3 small tasks
        task = {
            "name": f"Batch Task {i}",
            "description": f"Optimized batch processing task {i}",
            "executors": [
                {
                    "image": "alpine:latest",
                    "command": [
                        "sh", "-c", f"""
echo "Processing batch {i}..."
sleep 2
echo "Batch {i} completed at $(date)"
echo "batch_{i}_result" > /tmp/batch_{i}_output.txt
"""
                    ],
                    "workdir": "/tmp",
                    "stdout": f"/tmp/batch_{i}_stdout.log",
                    "stderr": f"/tmp/batch_{i}_stderr.log"
                }
            ],
            "outputs": [
                {
                    "name": f"batch_{i}_output",
                    "url": f"file:///tmp/batch_{i}_output.txt",
                    "path": f"/tmp/batch_{i}_output.txt"
                }
            ],
            "resources": {
                "cpu_cores": 1,
                "ram_gb": 0.5,
                "disk_gb": 0.5
            },
            "tags": {
                "batch_id": "demo_batch",
                "task_number": str(i),
                "batch_size": "3"
            }
        }
        batch_tasks.append(task)
    
    print(f"📤 Submitting {len(batch_tasks)} batch tasks...")
    
    batch_task_ids = []
    for i, task in enumerate(batch_tasks):
        task_id = submit_task(task)
        if task_id:
            batch_task_ids.append(task_id)
            print(f"  ✓ Batch task {i+1} submitted: {task_id}")
        else:
            print(f"  ❌ Failed to submit batch task {i+1}")
    
    if batch_task_ids:
        print(f"\n🔎 Monitoring {len(batch_task_ids)} batch tasks...")
        
        # Monitor all tasks in parallel
        completed_tasks = 0
        max_checks = 15
        
        for check in range(max_checks):
            all_complete = True
            
            for task_id in batch_task_ids:
                task_info = get_task_info(task_id)
                if task_info:
                    state = task_info.get('state')
                    if state not in ['COMPLETE', 'EXECUTOR_ERROR', 'SYSTEM_ERROR', 'CANCELED']:
                        all_complete = False
            
            if all_complete:
                completed_tasks = len(batch_task_ids)
                break
            
            time.sleep(2)
        
        print(f"📈 Batch processing complete: {completed_tasks}/{len(batch_task_ids)} tasks finished")
        
        # Show final status of all batch tasks
        print("\n📋 Batch Task Summary:")
        for i, task_id in enumerate(batch_task_ids):
            task_info = get_task_info(task_id)
            if task_info:
                state = task_info.get('state', 'UNKNOWN')
                print(f"  Task {i+1} ({task_id}): {state}")

def show_monitoring_strategies():
    """
    Demonstrate different monitoring strategies for various scenarios
    """
    print("\n👁️  Monitoring Strategies")
    print("=" * 30)
    
    strategies = [
        {
            "scenario": "Quick Tasks (< 1 minute)",
            "strategy": "Poll every 5-10 seconds, timeout after 2 minutes",
            "code": "monitor_task_progress(task_id, max_checks=12, check_interval=10)"
        },
        {
            "scenario": "Medium Tasks (1-30 minutes)",
            "strategy": "Poll every 30 seconds, timeout after 45 minutes",
            "code": "monitor_task_progress(task_id, max_checks=90, check_interval=30)"
        },
        {
            "scenario": "Long Tasks (> 30 minutes)",
            "strategy": "Poll every 2 minutes, use background monitoring",
            "code": "monitor_task_progress(task_id, max_checks=60, check_interval=120)"
        },
        {
            "scenario": "Batch Processing",
            "strategy": "Monitor subset, use task tags for grouping",
            "code": "# Monitor batch completion percentage using tags"
        }
    ]
    
    for strategy in strategies:
        print(f"\n🎯 {strategy['scenario']}:")
        print(f"  Strategy: {strategy['strategy']}")
        print(f"  Code: {strategy['code']}")

# Run best practices demonstrations
print("🎓 TES Best Practices and Optimization Guide")
print("=" * 50)

demonstrate_optimal_task_design()
show_performance_tips()
demonstrate_batch_optimization()
show_monitoring_strategies()

print("\n✨ Summary: Best Practices Checklist")
print("=" * 40)
checklist = [
    "✓ Use appropriate resource allocations",
    "✓ Design tasks with proper error handling",
    "✓ Implement progress reporting",
    "✓ Choose optimal container images",
    "✓ Use meaningful task names and tags",
    "✓ Monitor tasks with appropriate intervals",
    "✓ Handle outputs and logs properly",
    "✓ Design for scalability and efficiency"
]

for item in checklist:
    print(f"  {item}")

print("\n🎉 Best practices demonstration complete!")

## 9. Conclusion and Next Steps

This comprehensive notebook has demonstrated the full capabilities of the GA4GH Task Execution Service (TES) API through proTES, covering everything from basic task submission to advanced analytics and error handling.

### What We've Covered:

1. **Service Discovery** - Understanding TES service capabilities and configuration
2. **Task Creation & Submission** - Building and submitting various types of computational tasks
3. **Task Monitoring** - Real-time tracking of task execution and state transitions
4. **Advanced Examples** - Multi-step workflows, parallel processing, and complex task scenarios
5. **Analytics & Reporting** - Performance analysis, resource utilization, and task trends
6. **Middleware Configuration** - Leveraging proTES's extensible middleware system
7. **Error Handling** - Troubleshooting common issues and implementing robust error recovery
8. **Best Practices** - Optimization strategies for performance and reliability

### Key TES/proTES Features Demonstrated:

- ✓ **GA4GH Compliance** - Standard TES API endpoints and data models
- ✓ **Container Orchestration** - Docker-based task execution with resource management
- ✓ **Async Processing** - Non-blocking task submission with Celery/RabbitMQ backend
- ✓ **Monitoring & Logging** - Comprehensive task state tracking and output capture
- ✓ **Scalability** - Batch processing and parallel task execution
- ✓ **Extensibility** - Middleware system for custom workflow integration
- ✓ **Error Recovery** - Robust error handling and troubleshooting capabilities

### Next Steps:

1. **Production Deployment** - Scale proTES for production workloads with Kubernetes
2. **Custom Middleware** - Develop domain-specific middleware for your use cases
3. **Integration** - Connect TES with existing workflow management systems
4. **Monitoring** - Implement comprehensive monitoring and alerting for production
5. **Security** - Add authentication, authorization, and secure data handling

This notebook serves as both a learning resource and a practical reference for implementing TES-based computational workflows. The examples can be adapted for real-world bioinformatics, data processing, and scientific computing applications.